# Banana Ripeness — Unsupervised Clustering

Dataset: Kaggle `shahriar26s/banana-ripeness-classification-dataset` (overripe / ripe / rotten / unripe).

| Section | What |
|---|---|
| **0. Setup** | imports, config, helpers |
| **1. EDA** | balance, augmentation copies, near-duplicates / leakage, filename prefix vs label, backgrounds |
| **2. Prep** | pool splits → group by photo → drop label conflicts → dedupe → background removal |
| **3. Main model** | DINOv3 CLS + colour histogram → KMeans (k=4); no labels used |
| **4. Validation** | majority-vote mapping, metrics, confusion, error analysis, ablation, sweeps |

# 0. Setup

In [ ]:
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import kagglehub as kh
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from rembg import new_session, remove
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix, normalized_mutual_info_score, silhouette_score
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import v2

In [ ]:
# Paths
DATASET_SLUG = "shahriar26s/banana-ripeness-classification-dataset"
DATASET_DIR_NAME = "Banana Ripeness Classification Dataset"
WORK_DIR = Path("/tmp/bananafp")
DATASET_DIR = WORK_DIR / DATASET_DIR_NAME  # raw staged copy
CLEAN_DIR = WORK_DIR / "clean"  # deduped pool: <class>/<file>
NOBG_DIR = WORK_DIR / "clean_nobg"  # bg-removed copy of CLEAN_DIR (cached)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
UUID_RE = re.compile(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}")
PREFIX_RE = re.compile(r"musa-acuminata-([a-z]+)-")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

# Dedupe
HASH_SIZE = 16  # dHash -> 16*16 = 256 bits
HASH_MAX_DIST = 10  # Hamming <= this -> same photo

# DINOv3 backbone (local torch.hub cache)
HUB_DIR = Path.home() / ".cache/torch/hub"
DINO_REPO = HUB_DIR / "facebookresearch_dinov3_main"
DINO_MODEL = "dinov3_vits16plus"
DINO_WEIGHTS = HUB_DIR / "checkpoints/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"
DINO_RES = 224  # multiple of patch size 16

# Colour histogram
COLOR_RES = 128  # downscale before histogramming
WHITE_THRESH = 235  # all RGB channels above this -> background (after rembg)
H_BINS, S_BINS, V_BINS = 24, 8, 8

# Final model
FINAL_W = 0.4  # colour block weight vs. unit-norm DINO CLS block
FINAL_K = 4  # overripe / ripe / rotten / unripe

In [ ]:
def list_images(root: Path) -> list[Path]:
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS)

def prefix_of(p: Path) -> str:
    """Filename source prefix, e.g. 'freshunripe' (colour stage / source, not the label)."""
    m = PREFIX_RE.match(p.name)
    return m.group(1) if m else ""

def uuid_of(p: Path) -> str:
    """Source-photo id; Roboflow augmentations of one photo share it."""
    m = UUID_RE.search(p.name)
    return m.group() if m else p.name.split(".rf.")[0]

def dhash(p: Path, n: int = HASH_SIZE) -> np.ndarray:
    a = np.asarray(Image.open(p).convert("L").resize((n + 1, n)), dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).flatten()

def near_dup_pairs(paths, max_dist: int = HASH_MAX_DIST):
    """All (i, j, hamming) pairs with dHash Hamming distance <= max_dist."""
    H = np.array([dhash(p) for p in paths], dtype=np.uint8)
    pairs = []
    for i in range(len(H)):
        d = (H[i + 1 :] != H[i]).sum(1)
        pairs += [(i, i + 1 + int(j), int(d[j])) for j in np.nonzero(d <= max_dist)[0]]
    return pairs

def l2n(X: np.ndarray) -> np.ndarray:
    return X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-9)

def grid(rows: dict, n: int = 8, seed: int = SEED, size: float = 1.5):
    """rows: {row title: list of paths} -> n random images per row."""
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(rows), n, figsize=(size * n, size * 1.1 * len(rows)), squeeze=False)
    for ax_row, (title, ps) in zip(axes, rows.items()):
        for ax in ax_row:
            ax.axis("off")
        for ax, i in zip(ax_row, rng.choice(len(ps), size=min(n, len(ps)), replace=False)):
            ax.imshow(Image.open(ps[i]))
        ax_row[0].set_title(title, fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

# 1. EDA

In [ ]:
# Download (kagglehub cache) and stage a working copy once
if not DATASET_DIR.exists():
    cache = Path(kh.dataset_download(DATASET_SLUG))
    shutil.copytree(cache / DATASET_DIR_NAME if (cache / DATASET_DIR_NAME).exists() else cache, DATASET_DIR)

raw_paths = list_images(DATASET_DIR)
raw = pd.DataFrame({
    "path": raw_paths,
    "split": [p.relative_to(DATASET_DIR).parts[0] for p in raw_paths],
    "label": [p.parent.name for p in raw_paths],
    "prefix": [prefix_of(p) for p in raw_paths],
    "uuid": [uuid_of(p) for p in raw_paths],
})
print(len(raw), "images in", DATASET_DIR)
raw.head()

## 1.1 Class balance per split

In [ ]:
counts = pd.crosstab(raw["label"], raw["split"], margins=True)
display(counts)
counts.drop(index="All", columns="All").plot.bar(figsize=(6, 3), rot=0, title="images per class / split"); plt.show()
print("sizes:", Counter(Image.open(p).size for p in raw_paths).most_common(3))

In [ ]:
grid({c: list(g["path"]) for c, g in raw.groupby("label")})

## 1.2 Augmentation copies
Filenames are `<source>-<uuid>_jpg.rf.<hash>.jpg`; files sharing a UUID are Roboflow-augmented copies of one photo.

In [ ]:
raw["group_size"] = raw["uuid"].map(raw.groupby("uuid").size())
per_photo = raw.drop_duplicates("uuid")
print(len(raw), "files ->", len(per_photo), "unique UUIDs")
pd.crosstab(per_photo["split"], per_photo["group_size"])

In [ ]:
# 5 train photos, their 3 copies side by side
triples = [g["path"].tolist() for _, g in raw[raw["group_size"] == 3].groupby("uuid")]
rng = np.random.default_rng(SEED)
grid({f"photo {i}": triples[i] for i in rng.choice(len(triples), 5, replace=False)}, n=3, size=2)

In [ ]:
# how much does augmentation shift colour? (mean H/S/V per image)
def mean_hsv(p):
    return np.asarray(Image.open(p).convert("HSV"), dtype=float).reshape(-1, 3).mean(0)

sample = [triples[i] for i in rng.choice(len(triples), 300, replace=False)]
within = np.array([np.ptp([mean_hsv(p) for p in t], axis=0) for t in sample])
across = np.array([mean_hsv(t[0]) for t in sample]).std(0)
pd.DataFrame({"within-photo range (median)": np.median(within, 0), "across-photo std": across}, index=["H", "S", "V"]).round(1)

## 1.3 Near-duplicates across UUIDs / splits
dHash (256 bits) on every image; pairs within Hamming ≤ `HASH_MAX_DIST` are the same photo. UUIDs alone miss re-uploads.

In [ ]:
near_pairs = near_dup_pairs(raw_paths)
pp = pd.DataFrame(near_pairs, columns=["i", "j", "dist"])
pp["same_uuid"] = raw["uuid"].values[pp["i"]] == raw["uuid"].values[pp["j"]]
pp["same_label"] = raw["label"].values[pp["i"]] == raw["label"].values[pp["j"]]
pp["splits"] = ["-".join(sorted({raw["split"][i], raw["split"][j]})) for i, j in zip(pp["i"], pp["j"])]
print(len(pp), "near-identical pairs")
pd.crosstab([pp["same_uuid"], pp["same_label"]], pp["splits"], margins=True)

In [ ]:
def pair_grid(sel, title):
    fig, axes = plt.subplots(len(sel), 2, figsize=(4, 2 * len(sel)), squeeze=False)
    for row, (i, j) in zip(axes, sel):
        for ax, k in zip(row, (i, j)):
            ax.imshow(Image.open(raw_paths[k])); ax.axis("off")
            ax.set_title(f"{raw['split'][k]}/{raw['label'][k]}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

cross = pp[pp["splits"].str.contains("-")].sample(4, random_state=SEED)
pair_grid(list(zip(cross["i"], cross["j"])), "same photo, different split (leakage)")
conf = pp[~pp["same_label"]]
pair_grid(list(zip(conf["i"], conf["j"]))[:4], "same photo, different label")

## 1.4 Filename prefix vs. folder label

In [ ]:
pd.crosstab(raw["prefix"], raw["label"], margins=True)

In [ ]:
# rotten/ holds files named -ripe- / -unripe-: mislabels or damaged fruit?
grid({
    "rotten/ named -ripe-": list(raw.query("label == 'rotten' and prefix == 'ripe'")["path"]),
    "rotten/ named -unripe-": list(raw.query("label == 'rotten' and prefix == 'unripe'")["path"]),
    "rotten/ named -rotten-": list(raw.query("label == 'rotten' and prefix == 'rotten'")["path"]),
    "ripe/ named -ripe-": list(raw.query("label == 'ripe' and prefix == 'ripe'")["path"]),
}, n=6)

## 1.5 Backgrounds by source

In [ ]:
grid({p: list(g["path"]) for p, g in raw.groupby("prefix")}, n=6)

## EDA findings → prep decisions

| Finding | Prep action |
|---|---|
| Train files are Roboflow ×3 augmentations (same UUID); valid/test are single | Treat a UUID group as one photo |
| Same photo re-uploaded under new UUIDs, some across splits (leakage) and a few across classes | Merge near-identical photos (dHash), pool all splits |
| A few photo groups carry both `ripe` and `rotten` | Drop conflicting groups |
| Unsupervised task: no train/test split needed | One deduped pool, one file per photo |
| `rotten/` files named `-ripe-`/`-unripe-` are damaged fruit at that colour stage, not mislabels | Keep folder labels |
| Backgrounds differ strongly by source | Remove background before embedding |

# 2. Prep

## 2.1 Group by photo (UUID + near-identical pixels)

In [ ]:
# Union-find over raw file indices
parent = list(range(len(raw_paths)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def union(i, j):
    parent[find(i)] = find(j)

for ids in raw.groupby("uuid").indices.values():
    for j in ids[1:]:
        union(ids[0], j)
for i, j, _ in near_pairs:  # from EDA 1.3
    union(i, j)

groups = defaultdict(list)
for i in range(len(raw_paths)):
    groups[find(i)].append(i)
groups = list(groups.values())

split_of, label_of = raw["split"].to_numpy(), raw["label"].to_numpy()
print(len(groups), "photo groups")
print("groups spanning >1 split:", sum(len(set(split_of[g])) > 1 for g in groups))

## 2.2 Drop label conflicts

In [ ]:
conflicts = [g for g in groups if len(set(label_of[g])) > 1]
for g in conflicts:
    print(sorted(set(label_of[g])), len(g), "files")
groups = [g for g in groups if len(set(label_of[g])) == 1]
print(f"dropped {len(conflicts)} conflicting groups -> {len(groups)} left")

## 2.3 Dedupe → clean pool

In [ ]:
# One representative per group (random, seeded; all train copies are augmented, none is "the original")
rng = random.Random(SEED)
keep = [(rng.choice(g), len(g)) for g in groups]

if CLEAN_DIR.exists():
    shutil.rmtree(CLEAN_DIR)
for i, _ in keep:
    dst = CLEAN_DIR / label_of[i] / raw_paths[i].name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_paths[i], dst)

print(len(raw_paths), "->", len(keep), "kept at", CLEAN_DIR)

## 2.4 Background removal

In [ ]:
# default bria-rmbg (~1GB) OOMs -> tiny u2netp, one shared session
REM_BG_SESSION = new_session("u2netp")

def remove_bg(pil: Image.Image) -> Image.Image:
    fg = remove(pil.convert("RGB"), session=REM_BG_SESSION)
    bg = Image.new("RGB", fg.size, (255, 255, 255))
    bg.paste(fg, mask=fg.split()[3])
    return bg

# cached: only files missing from NOBG_DIR are processed
for i, _ in keep:
    q = NOBG_DIR / label_of[i] / raw_paths[i].name
    if not q.exists():
        q.parent.mkdir(parents=True, exist_ok=True)
        remove_bg(Image.open(CLEAN_DIR / label_of[i] / raw_paths[i].name)).save(q)
print("bg-removed ->", NOBG_DIR)

## 2.5 Clean pool + sanity checks

In [ ]:
pool = pd.DataFrame([{
    "path": NOBG_DIR / label_of[i] / raw_paths[i].name,
    "clean_path": CLEAN_DIR / label_of[i] / raw_paths[i].name,
    "label": label_of[i],
    "prefix": raw["prefix"][i],
    "orig_split": split_of[i],
    "augmented": split_of[i] == "train",  # Roboflow only augmented train
    "group_size": n,
} for i, n in keep]).sort_values("path", ignore_index=True)

assert pool["path"].is_unique, "filename collision in clean pool"
assert len(list_images(CLEAN_DIR)) == len(list_images(NOBG_DIR)) == len(pool)
pd.crosstab(pool["label"], pool["orig_split"], margins=True)

In [ ]:
# eyeball: original vs bg-removed
sample = pool.sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for col, (_, r) in enumerate(sample.iterrows()):
    axes[0, col].imshow(Image.open(r["clean_path"])); axes[0, col].set_title(r["label"], fontsize=8)
    axes[1, col].imshow(Image.open(r["path"]))
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

# 3. Main model: DINOv3 CLS + colour → KMeans

Unsupervised: no labels are used in this section.
- **DINOv3 ViT-S+/16** backbone, CLS token (384-d). Self-supervised, so it keeps colour / texture / damage cues that an ImageNet-label backbone (ConvNeXt) collapses into "banana".
- **Colour**: H/S/V histograms over banana pixels only (40-d); bg is white after rembg, so mask = not near-white.
- **Feature** = `[l2n(dino_cls), FINAL_W * l2n(colour)]` → **KMeans** `k = FINAL_K`.

In [ ]:
@torch.no_grad()
def embed(paths, model_fn, transform, batch_size: int = 64) -> np.ndarray:
    vecs = []
    for i in range(0, len(paths), batch_size):
        batch = torch.stack([transform(Image.open(p).convert("RGB")) for p in paths[i : i + batch_size]])
        vecs.append(model_fn(batch.to(DEVICE)).float().cpu())
    return torch.cat(vecs).numpy()

def color_hist(p: Path) -> np.ndarray:
    """H/S/V histograms over banana pixels only; each sums to 1."""
    img = Image.open(p).convert("RGB").resize((COLOR_RES, COLOR_RES))
    mask = ~(np.asarray(img) > WHITE_THRESH).all(-1)
    hsv = np.asarray(img.convert("HSV"))[mask]
    if len(hsv) == 0:
        return np.zeros(H_BINS + S_BINS + V_BINS)
    hists = [np.histogram(hsv[:, i], bins=n, range=(0, 256))[0] for i, n in enumerate((H_BINS, S_BINS, V_BINS))]
    return np.concatenate([h / h.sum() for h in hists])

def combine(X_dino_cls, X_color, w: float = FINAL_W) -> np.ndarray:
    return np.hstack([l2n(X_dino_cls), w * l2n(X_color)])

## 3.1 Features

In [ ]:
paths = list(pool["path"])

# DINOv3 backbone only (no head); LVD-1689M weights -> ImageNet eval transform (DINOv3 README)
dino = torch.hub.load(str(DINO_REPO), DINO_MODEL, source="local", weights=str(DINO_WEIGHTS)).eval().to(DEVICE)
dino_tf = v2.Compose([
    v2.ToImage(),
    v2.Resize((DINO_RES, DINO_RES), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
X_dino_cls = embed(paths, lambda x: dino.forward_features(x)["x_norm_clstoken"], dino_tf)
print("DINO CLS:", X_dino_cls.shape)

In [ ]:
X_color = np.array([color_hist(p) for p in paths])
X_final = combine(X_dino_cls, X_color, FINAL_W)
print("colour:", X_color.shape, "| final feature:", X_final.shape, f"(w={FINAL_W})")

## 3.2 KMeans

In [ ]:
kmeans = KMeans(n_clusters=FINAL_K, n_init=10, random_state=SEED).fit(X_final)
clusters = kmeans.labels_
print("cluster sizes:", np.bincount(clusters))

In [ ]:
# what each cluster looks like (no labels)
grid({f"cluster {c}": [paths[i] for i in np.flatnonzero(clusters == c)] for c in range(FINAL_K)})

# 4. Validation

- Clusters are named by **majority vote** of folder labels (labels used only here).
- Metrics are on the same pool KMeans was fit on → report **NMI / ARI** as cluster quality, not held-out accuracy.
- **Silhouette** is label-free; it's shown next to ARI in the sweeps to check `w` / `k` without labels.

In [ ]:
y = pool["label"].to_numpy()
prefix = pool["prefix"].to_numpy()
augmented = pool["augmented"].to_numpy()
classes = sorted(set(y))

def majority_map(clusters):
    """cluster id -> most common true label in it"""
    return {c: pd.Series(y[clusters == c]).mode()[0] for c in np.unique(clusters)}

def score(X, clusters):
    c2l = majority_map(clusters)
    y_pred = np.array([c2l[c] for c in clusters])
    return {
        "acc": (y_pred == y).mean(),
        "acc_not_aug": (y_pred == y)[~augmented].mean(),
        "acc_aug": (y_pred == y)[augmented].mean(),
        "NMI": normalized_mutual_info_score(y, clusters),
        "ARI": adjusted_rand_score(y, clusters),
        "silhouette": silhouette_score(X, clusters),  # label-free
        "largest_cluster": np.bincount(clusters).max() / len(y),
        "labels_covered": len(set(c2l.values())),
        **{f"rec_{c}": (y_pred[y == c] == c).mean() for c in classes},
    }, y_pred

def fit_score(X, k=FINAL_K, seed=SEED):
    return score(X, KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X))

## 4.1 Final model metrics

In [ ]:
final, y_pred = score(X_final, clusters)
print("cluster -> label:", majority_map(clusters))
pd.Series(final).round(3)

In [ ]:
cm = confusion_matrix(y, y_pred, labels=classes)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(classes)), classes, rotation=45); ax.set_yticks(range(len(classes)), classes)
ax.set_xlabel("predicted (cluster majority)"); ax.set_ylabel("true (folder)")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()

## 4.2 Error analysis

In [ ]:
# rows: folder label + filename source prefix, cols: predicted
pd.crosstab([y, prefix], y_pred, rownames=["true", "prefix"], colnames=["pred"], margins=True)

In [ ]:
# samples per predicted cluster (title = true label, red = mismatch)
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(classes), 8, figsize=(12.8, 1.8 * len(classes)))
for row, c in zip(axes, classes):
    for ax, i in zip(row, rng.choice(np.flatnonzero(y_pred == c), 8, replace=False)):
        ax.imshow(Image.open(paths[i])); ax.axis("off")
        ax.set_title(y[i], fontsize=7, color="black" if y[i] == c else "red")
    row[0].text(-0.15, 0.5, f"pred: {c}", transform=row[0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# PCA: true labels vs clusters
Z = PCA(n_components=2, random_state=SEED).fit_transform(X_final)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, lab, title in [(axes[0], y, "true label"), (axes[1], y_pred, "cluster (majority label)")]:
    for c in classes:
        m = lab == c
        ax.scatter(Z[m, 0], Z[m, 1], s=3, alpha=0.5, label=c)
    ax.set_title(title); ax.legend(markerscale=4, fontsize=8)
plt.tight_layout(); plt.show()

## 4.3 Ablation: which features?
All at `k=4`. ConvNeXt-Tiny (conv stages + avg pool, no head) is the ImageNet-label baseline, on the same bg-removed images.

In [ ]:
cnx_w = ConvNeXt_Tiny_Weights.DEFAULT
_cnx = convnext_tiny(weights=cnx_w)
convnext = torch.nn.Sequential(_cnx.features, _cnx.avgpool, torch.nn.Flatten(1)).eval().to(DEVICE)
X_convnext = embed(paths, convnext, cnx_w.transforms())

ablation = {
    "convnext": l2n(X_convnext),
    "dino_cls": l2n(X_dino_cls),
    "color": l2n(X_color),
    "convnext+color": combine(X_convnext, X_color, FINAL_W),
    f"dino_cls+color (final, w={FINAL_W})": X_final,
}
cols = ["acc", "NMI", "ARI", "silhouette", "largest_cluster", "rec_overripe", "rec_ripe", "rec_rotten", "rec_unripe"]
pd.DataFrame({k: fit_score(X)[0] for k, X in ablation.items()}).T[cols].round(3)

## 4.4 Sweep: colour weight `w`
3 KMeans seeds per weight (mean ± std). Too little colour → unripe collapses into ripe; too much → rotten gets sorted by colour instead of damage.

In [ ]:
rows = []
for w in [0.25, 0.35, 0.4, 0.45, 0.5, 0.6, 0.75, 1.0]:
    for s in [0, 1, 2]:
        rows.append({"w": w, "seed": s, **fit_score(combine(X_dino_cls, X_color, w), seed=s)[0]})
pd.DataFrame(rows).groupby("w")[["acc", "NMI", "ARI", "silhouette", "rec_rotten", "rec_unripe"]].agg(["mean", "std"]).round(3)

## 4.5 Sweep: number of clusters `k`
Majority-vote `acc` rises with `k` almost automatically (smaller clusters are purer); compare ARI and silhouette.

In [ ]:
sweep_k = pd.DataFrame({k: fit_score(X_final, k=k)[0] for k in [3, 4, 5, 6, 8, 12]}).T.rename_axis("k")
sweep_k[["acc", "NMI", "ARI", "silhouette", "labels_covered", "rec_rotten", "rec_unripe"]].round(3)

## Conclusions
Fill in after running:
- Final model (DINOv3 CLS + colour, KMeans k=4): acc / NMI / ARI.
- DINOv3 vs ConvNeXt, and what colour adds (ablation).
- Augmented vs not (`acc_aug` vs `acc_not_aug`).
- Remaining error: `rotten` is damage, not a colour stage, so damaged green / yellow / brown fruit falls into the unripe / ripe / overripe clusters.